In [13]:
import matplotlib.pyplot as plt
import parselib
import sys
import seaborn as sns
import numpy as np
import matplotlib as mpl
import plotconfig
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

In [ ]:
bbr_version = "bbr3"

args = {
    "chaos": "on_1",
    "timestamp": ["20251119", "2025120", "20251", "2025112"],
    "cca": ["cubic", bbr_version],
    "test_cca": "yes",
    "kernel": ["kernel6-1", "zkernel6-13-BBRv3"],
    "loss_mode": "none",
    "n": [30],
    "parallel": [1],
    "bdp": [1],
    "rate": [1000, 500, 100, 10],
    "delay_rtt": [10],
    "deadline_run": [1000000,10000000,20000000],
}
if bbr_version == "bbr":
    args["kernel"] = ["kernel6-1"]

metric = "bits_per_second"

baselogpath = "../data"

In [15]:
# create filter and populate it from args dictionary
fil = parselib.Filter()
fil.fill_from_dict(args)
df = parselib.logs_to_df(baselogpath, fil)
if df is None:
    sys.exit()

start 7200
end 7200
bytes 7200
bits_per_second 7200
mbps_timeseries 7200
rttms_timeseries 7200
retransmits 7200
timestamp 7200
iteration 7200
cpu_host_total 7200
cpu_host_user 7200
cpu_host_system 7200
cpu_remote_total 7200
chaos 7200
deadline_run 7200
deadline_period 7200
os 7200
bdp 7200
setup 7200
cca 7200
cpus 7200
kernel 7200
mode 7200
loss 7200
rate 7200
delay_rtt 7200
buffer_size_bytes 7200
parallel 7200
socket_buffer 7200
app_buffer 7200
n 7200
sysctl_cmd 7200
vm 7200
bandwidth_delay_product 7200
loss_mode 7200
vms 7200
pacing 7200
hyperthreading 7200
tso 7200
qdisc 7200
hpet 7200
tsc 7200
hostq 7200
loadperc 7200
deadline_period_factor 7200
random_loss_rate 7200
gemodel_q 7200
original_cca 7200
test_cca 7200
default_qdisc 7200
json 7200


In [16]:
df["slice_perc"] = (df["deadline_run"]/df["deadline_period"])*100
df["mbps"] = df["bits_per_second"]/1000000

replace_label = {
    "bbr": "BBRv1",
    "bbr2": "BBRv2",
    "bbr3": "BBRv3",
    "cubic": "Cubic"
}

In [17]:
def strip(df, savefig = False):    
    if savefig:
        mpl.use('agg')
        plotconfig.configure_conext()
        width = plotconfig.pt2inch(plotconfig.COLUMN_WIDTH)
        height = width*(1/3)
        FIG_SIZE = (width, height)
    
    paletti = plotconfig.COLORS[:5]
    fig, axs = plt.subplots(1,3,figsize=FIG_SIZE, sharey=True,constrained_layout=True)


    for it, runti in enumerate(sorted(df["deadline_run"].unique())):
        data = df[df["deadline_run"] == runti]
        data = data.sort_values(by=['deadline_run'])

        df_cubic = data[data["cca"] == "cubic"]
        df_cubic = df_cubic[df_cubic["kernel"] == "kernel6-1"]
       
        df_cubic = df_cubic[["mbps","slice_perc","delay_rtt","rate"]].groupby(["slice_perc", "delay_rtt", "rate"],as_index=False).median()
        for i_ in args["delay_rtt"]:
            marker = "o"
            sns.stripplot(ax=axs[it], data=df_cubic[df_cubic["delay_rtt"] == i_], x="slice_perc", y="mbps", hue="rate", legend=False, jitter=False, palette=paletti, marker=marker, size=1.75, edgecolor='black',linewidth=0.5)
        if it != 1:
            axs[it].set_xlabel("")

        cca = bbr_version
        data_sorted=data[data["cca"] == cca]
        
        data_sorted = data_sorted[data_sorted["kernel"] == args["kernel"][-1]]
        
        for i_ in args["delay_rtt"]:
            alph = 0.3
            marker = "o"
            sns.stripplot(ax=axs[it], data=data_sorted[data_sorted["delay_rtt"] == i_], x="slice_perc", y="mbps", hue="rate", jitter=True, alpha=alph, zorder=0, palette=paletti, dodge=True, legend=True, marker = marker, size=3)
            sns.boxplot(ax=axs[it], data=data_sorted[data_sorted["delay_rtt"] == i_], x="slice_perc", y="mbps", hue="rate", palette=paletti,fliersize=0,legend=False)

        axs[it].set_xticks([0,1,2,3,4,5,6,7,8], ["10","15","20","25","30","35","40","45","50"], fontsize=plotconfig.FONT_SIZE-3)
        axs[it].tick_params(axis='both', which='major', labelsize=plotconfig.FONT_SIZE-3)
        axs[it].vlines([0.5,1.5,2.5,3.5,4.5,5.5,6.5,7.5], ymin=-5, ymax = int(args["rate"][0])+5, color="gainsboro", linewidth=0.5)
        axs[it].set_xlim(-0.5, 8.5)
    
        axs[it].set_title(f"Timeslice {int(runti/1000000)}ms",fontsize=plotconfig.FONT_SIZE-3,pad=3)
        if runti == 1000000:
            axs[it].set_xlabel("")
            leg1=axs[it].legend(handles = [
                    Line2D([0], [0], linestyle='none', mfc=paletti[2], mec="black", mew=0.1, marker="o", markersize=3, label=replace_label[cca]),
                    Line2D([0], [0], linestyle='none', mfc=paletti[2], mec="black", mew=0.6, marker="o", markersize=1.85, label='Cubic'),

                ],loc="lower left", framealpha=0.5,title_fontsize=plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3)
            axs[it].add_artist(leg1)
            h, l = axs[it].get_legend_handles_labels()
            for hii in h:
                hii.set_alpha(1)
            axs[it].legend(handles = h, title = "bw [mbps]", loc="lower right", framealpha=0.5,title_fontsize=plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3)
        elif runti == 10000000:
            axs[it].set_xlabel("VM CPU share [\%]",fontsize=plotconfig.FONT_SIZE-3)
            axs[it].legend([])
        elif runti == 20000000:
            axs[it].set_xlabel("")
            axs[it].legend([])

    axs[0].set_ylabel("Avg. throughput [Mbps]",fontsize=plotconfig.FONT_SIZE-3)
    axs[0].set_yscale('log')
    
    if savefig:
        fig.savefig(f"figures/figure_{'7' if bbr_version == 'bbr3' else '14'}.pdf", format="pdf")
    else:
        plt.show()

<>:57: SyntaxWarning: invalid escape sequence '\%'
<>:57: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_241142/4097365254.py:57: SyntaxWarning: invalid escape sequence '\%'
  axs[it].set_xlabel("VM CPU share [\%]",fontsize=plotconfig.FONT_SIZE-3)


In [18]:
strip(df, True)

/tmp/ipykernel_241142/4097365254.py:23: UserWarning: The palette list has more values (5) than needed (4), which may not be intended.
  sns.stripplot(ax=axs[it], data=df_cubic[df_cubic["delay_rtt"] == i_], x="slice_perc", y="mbps", hue="rate", legend=False, jitter=False, palette=paletti, marker=marker, size=1.75, edgecolor='black',linewidth=0.5)
/tmp/ipykernel_241142/4097365254.py:35: UserWarning: The palette list has more values (5) than needed (4), which may not be intended.
  sns.stripplot(ax=axs[it], data=data_sorted[data_sorted["delay_rtt"] == i_], x="slice_perc", y="mbps", hue="rate", jitter=True, alpha=alph, zorder=0, palette=paletti, dodge=True, legend=True, marker = marker, size=3)
/tmp/ipykernel_241142/4097365254.py:36: UserWarning: The palette list has more values (5) than needed (4), which may not be intended.
  sns.boxplot(ax=axs[it], data=data_sorted[data_sorted["delay_rtt"] == i_], x="slice_perc", y="mbps", hue="rate", palette=paletti,fliersize=0,legend=False)
/tmp/ipyke